In [2]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.window import *

spark = SparkSession.builder.appName("ManufacturingParts").getOrCreate()

# Create df1
df1 = spark.createDataFrame(
    [
        ("P1", "2023-01-01", "Location_A"),
        ("P2", "2023-01-02", "Location_B"),
        ("P3", "2023-01-03", "Location_C"),
    ],
    ["product_id", "manufacturing_date", "manufacturing_location"],
)

# Create df2
df2 = spark.createDataFrame(
    [
        ("P1", "Widget_A", "Widget"),
        ("P2", "Gadget_B", "Gadget"),
        ("P3", "Device_C", "Device"),
    ],
    ["product_id", "product_name", "product_type"],
)

df1.show()
df2.show()

+----------+------------------+----------------------+
|product_id|manufacturing_date|manufacturing_location|
+----------+------------------+----------------------+
|        P1|        2023-01-01|            Location_A|
|        P2|        2023-01-02|            Location_B|
|        P3|        2023-01-03|            Location_C|
+----------+------------------+----------------------+

+----------+------------+------------+
|product_id|product_name|product_type|
+----------+------------+------------+
|        P1|    Widget_A|      Widget|
|        P2|    Gadget_B|      Gadget|
|        P3|    Device_C|      Device|
+----------+------------+------------+



In [5]:
window_spec = Window.orderBy("product_id")

df1.join(broadcast(df2), "product_id", "inner").withColumn(
    "row_number", row_number().over(window_spec)
).select(
    "manufacturing_date",
    "manufacturing_location",
    "product_id",
    "product_name",
    "product_type",
    "row_number",
).show()

+------------------+----------------------+----------+------------+------------+----------+
|manufacturing_date|manufacturing_location|product_id|product_name|product_type|row_number|
+------------------+----------------------+----------+------------+------------+----------+
|        2023-01-01|            Location_A|        P1|    Widget_A|      Widget|         1|
|        2023-01-02|            Location_B|        P2|    Gadget_B|      Gadget|         2|
|        2023-01-03|            Location_C|        P3|    Device_C|      Device|         3|
+------------------+----------------------+----------+------------+------------+----------+

